In [ ]:
import sys
# 1. Force setuptools to enable its internal local distutils backup
import os
os.environ["SETUPTOOLS_USE_DISTUTILS"] = "local"

# 2. Programmatically map the module hook into the Jupyter kernel's active memory
import setuptools
try:
    import _distutils_hack
    _distutils_hack.do_override()
except ImportError:
    pass

print("Jupyter Kernel environment patched successfully!")


#**Strategic Customer Retention: Leveraging Cohort Insights for ShopSphere Inc.'s Sustainable Growth**

This project involved a comprehensive analysis of e-commerce transactional data to gain insights into customer behavior and develop a customer segmentation strategy.

* **Key steps included:**

  * **Data Ingestion and Cleaning:** The dataset was loaded, basic information was inspected, and initial data quality checks confirmed no missing values. Duplicate entries were removed to ensure data integrity.

  * **Feature Engineering:** A Revenue column was created (Quantity * UnitPrice), and data validation confirmed all Quantity and UnitPrice values were positive.

  * **Monthly Revenue Trend Analysis:** The data was used to visualize monthly revenue trends, providing an overview of business performance over time.

  * **Cohort Analysis:** Customers were grouped into cohorts based on their first purchase month, and a retention heatmap was generated. This revealed initial customer drop-off patterns, subsequent stabilization, and varying retention rates across different cohorts.

  * **RFM (Recency, Frequency, Monetary) Analysis:** RFM metrics were calculated for each customer, quantifying their recent activity, purchase frequency, and total spending. These metrics were then scaled to prepare for clustering.


  * **K-Means Clustering:** The scaled RFM features were used to segment customers into four distinct groups using K-Means clustering. The optimal number of clusters was determined using the Elbow Method, Silhouette Score, and KElbowVisualizer.

  * **Customer Segmentation and Profiling:** Each of the four clusters was profiled based on their average RFM values and given meaningful names:

    * **Loyal Customers:** Moderate recency, high frequency, high monetary value.

    * **New/Occasional Customers:** Moderate recency, lower frequency, lower monetary value.

    * **At-Risk/Churned Customers:** High recency (less recent purchases), moderate frequency/monetary value.

    * **Champions/VIPs:** Excellent recency, very high frequency, highest monetary value.

* **Overall Outcome:** The project successfully identified distinct customer segments with actionable insights for targeted marketing campaigns, customer retention strategies, and optimizing customer lifetime value. An RFM cluster report was also saved to a CSV file.

# New Section

**Data Understanding and Ingestion**

The pd.read_csv() function is the primary tool in pandas for loading delimited data into a DataFrame. While it has dozens of parameters, they can be grouped by their specific purpose in the parsing process.



**Core Data Location & Format**

**filepath_or_buffer**: The path to your file (local string or URL) or a file-like object.

**sep / delimiter**: The character used to separate values (default is ','). delimiter is an alias for sep.

**engine**: The parser engine to use ('c', 'python', or 'pyarrow'). The C engine is faster, while the Python engine is more feature-complete.

**encoding**: The character encoding (e.g., 'utf-8', 'latin-1') used for the file.



**Column & Index Handling**

**header**: Specifies which row contains column names (default is 0). Use None if there is no header.

**names**: A list of names to use for columns, useful if the file lacks a header or you want to override it.

**index_col**: The column(s) to use as the row labels (index) of the DataFrame.

**usecols**: A list of specific columns to load, which saves memory by ignoring unwanted data.

**dtype**: A dictionary to force specific data types for specific columns (e.g., {'ID': str}).



**Row Selection & Filtering**

**skiprows**: The number of lines to skip at the start, or a list of specific row indices to ignore.

**skipfooter**: The number of lines to skip at the very bottom of the file.

**nrows**: The specific number of rows to read from the beginning of the file.

**skip_blank_lines**: If True, ignores empty lines instead of interpreting them as missing values.



**Handling Missing & Special Values**

**na_values**: Additional strings to recognize as NaN (e.g.['missing', 'N/A']).

**keep_default_na**: Whether to include the standard pandas NaN list (like '', 'NULL') alongside your na_values.

**na_filter**: If False, disables all missing value detection (can improve performance for clean datasets).

**true_values / false_values**: Lists of strings to treat as Boolean True or False.



**Date & Time Parsing**

**parse_dates**: Boolean or list indicating which columns should be converted to datetime objects.

**date_format**: Specifies the format of the date strings to speed up parsing (e.g., '%Y-%m-%d').

**dayfirst**: If True, parses dates like 10/01/2000 as 10th January.



**Memory & Performance**

**chunksize**: Returns an iterator that reads the file in "chunks" of a specified row count, essential for files larger than RAM.

**low_memory**: Process the file in chunks internally to reduce memory usage (default is True, but can lead to mixed-type warnings).

**memory_map**: If True, maps the file directly into memory for faster access on some systems.



**Error & Quoting Control**

**on_bad_lines**: Defines what to do with lines that have too many fields ('error', 'warn', or 'skip').

**quotechar / quoting**: Defines the character used for quoted fields and how the parser handles them.

**comment**: A character (like #) that tells the parser to ignore the rest of the line.




**Data Dictionary**

|   Column Name   |   Data Type   |   Description   |
|--------------------|--------------------|--------------------|
|   InvoiceNo   |   String   |   A unique 6-digit integral number assigned to each transaction.   |
|   StockCode   |   String   |   A 5-digit integral number uniquely identifying each product.   |
|   Description   |   String   |  Product name/description.   |
|   Quantity  |   Integer   |  The quantities of each product per transaction.   |
|   InvoiceDate  |   DateTime   |  The date and time when each transaction was generated.   |
|   UnitPrice  |   Float   |  The price per unit of the product in GBP.   |
|   CustomerID  |   String   |  A unique 5-digit integral number identifying each customer.   |
|   Country  |   String   |  The name of the country where the customer resides.   |

In [ ]:


from datetime import date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import is_integer_dtype, is_float_dtype
from scipy.stats import skew, kurtosis
from IPython.display import display, HTML


dtype = {'InvoiceNo': str, 'StockCode': str, 'Description': str, 'Quantity': int, 'UnitPrice': float, 'CustomerID': str, 'Country': str}

#df = pd.read_csv('/content/drive/MyDrive/Amdari/ShopSphere_Dataset.csv', sep=',', dtype=dtype, parse_dates=['InvoiceDate'])

df = pd.read_csv('ShopSphere_Dataset.csv', sep=',', dtype=dtype, parse_dates=['InvoiceDate'])

df.head()

df.columns

**Check Size of the Data and Data Information**

In [ ]:
print(f"Dataset Dimension {df.shape[0]} rows and {df.shape[1]} columns \n")
print(f"Total Elements: {df.size}\n")
print(f"Column Names: {df.columns.tolist()}\n")
print('Summary Data Information - Column Data Types \n')
df.info()
print('\n')

df.head()



**Check Missing Values and How They Are Distributed Across Variables**

In [ ]:
missing_column_counts = df.isna().sum()
missing_column_ratios = df.isna().mean()


if missing_column_counts.sum() == 0 and missing_column_ratios.mean() ==0:
  print('No Missing Values Found')
else:
  display(pd.DataFrame([missing_column_counts, missing_column_ratios], index=['Count', 'Ratio']).T)

**EXPLORING DATA CHARACTERISTICS**

*   **Understanding Data Quality**: Identifying outliers, and inconsistencies that might affect subsequent analysis.

*   **Data Exploration**: Quickly understand the cardinality of categorical variables (e.g., "How many unique cities are in this dataset?").

*   **Data Cleaning**: Identify columns that have only one unique value (zero variance), which are often useless for machine learning models.

*   **Feature Engineering**: Check for potential primary key candidates by seeing if the nunique() count matches the total number of rows.

In [ ]:
likely_keys = []
constant_columns = []
near_zero_variance = []
time_stats = {}
stats = {}

for col in df.columns:
  if df[col].nunique() == df.shape[0]:
    likely_keys.append(col)
    continue
  if df[col].nunique() == 1:
    constant_columns.append(col)
    continue
  if df[col].nunique()/df.shape[0] >= 0.95:
    near_zero_variance.append(col)
    continue

title = "Likely-Key Analysis"

# 1. Capture initial state
initial_dim = f"{df.shape[0]} rows x {df.shape[1]} columns"
keys_count = len(likely_keys)

if likely_keys:
    # 2. Perform the drop
    df = df.drop(columns=likely_keys)
    status = "Keys Removed"
    dropped_cols = ", ".join(likely_keys)

else:
    status = "No Likely Keys Found"
    dropped_cols = "None"

# 3. Create the Summary DataFrame
likely_key_report = pd.DataFrame({
    "Analysis Type": ["Likely-Key Analysis"],
    "Status": [status],
    "Columns Identified": [dropped_cols],
    "Count Dropped": [keys_count],
    "Original Dimension": [initial_dim],
    "New Dimension": [f"{df.shape[0]} rows x {df.shape[1]} columns"]
})

display(HTML(f"<h3>{title}</h3>"))
display(likely_key_report)
print('\n\n')


title = "Constant Column Analysis"

# 1. Capture the initial state
initial_shape = f"{df.shape[0]} rows, {df.shape[1]} columns"
num_dropped = len(constant_columns)

if constant_columns:
    # 2. Perform the drop
    df = df.drop(columns=constant_columns)
    status = "Constant Columns Removed"
    col_list = ", ".join(constant_columns)

else:
    status = "No Constant Columns Found"
    col_list = "None"


# 3. Create the Summary DataFrame
constant_analysis_report = pd.DataFrame({
    "Analysis Type": ["Constant Column Analysis"],
    "Status": [status],
    "Columns Identified": [col_list],
    "Count Dropped": [num_dropped],
    "Original Dimension": [initial_shape],
    "New Dimension": [f"{df.shape[0]} rows, {df.shape[1]} columns"]
})

display(HTML(f"<h3>{title}</h3>"))
display(constant_analysis_report)
print('\n\n')


title = "Near Zero Variance Analysis"

# 1. Capture Initial State
initial_shape = df.shape
nzv_found = len(near_zero_variance) > 0

# 2. Process Dropping
if nzv_found:
    df = df.drop(columns=near_zero_variance)
    status = "Columns Dropped"
    dropped_list = ", ".join(near_zero_variance)

else:
    status = "No Columns Dropped"
    dropped_list = "None"


# 3. Build the Summary DataFrame
nzv_report = pd.DataFrame({
    "Analysis": ["Near Zero Variance"],
    "Status": [status],
    "Columns Identified": [dropped_list],
    "Count Dropped": [len(near_zero_variance)],
    "Original Shape": [f"{initial_shape[0]}x{initial_shape[1]}"],
    "New Shape": [f"{df.shape[0]}x{df.shape[1]}"]
})

display(HTML(f"<h3>{title}</h3>"))
display(nzv_report)
print('\n\n')

df_num = df.select_dtypes(include='number')
df_time = df.select_dtypes(include='datetime')
df_cat = df.select_dtypes(exclude=['number', 'datetime'])

title = "Cardinality Of Categorical Variables"


column_labels = {
    'InvoiceNo': 'Number of Transactions',
    'StockCode': 'Number of Products',
    'Description': 'Item Categories',
    'CustomerID': 'Number of Customers',
    'Country': 'Number of Countries'
}

cardinality_data = []

for col in df_cat.columns:
    label = column_labels.get(col, col)
    n_unique = df_cat[col].nunique()

    # Determine the Category Description
    if n_unique == 2:
        category = "Categorical - Binary"
    elif 2 < n_unique <= 10:
        category = "Categorical - Low Cardinality"
    else:
        category = "Categorical - High Cardinality"

    # Store the results in a list
    cardinality_data.append({
        "Variable": label,
        "Original Column": col,
        "Cardinality Type": category,
        "Unique Count": n_unique
    })

# Create and display the DataFrame
df_cardinality_report = pd.DataFrame(cardinality_data)

display(HTML(f"<h3>{title}</h3>"))
display(df_cardinality_report)
print('\n\n')


title = "Numerical Variable Analysis"
stats = {}

for col in df_num.columns:
    current_series = df_num[col]

    # 1. Determine Type and Description
    if is_integer_dtype(current_series):
        var_type = "Discrete (Integer)"
    elif is_float_dtype(current_series):
        var_type = "Continuous (Float)"
    else:
        var_type = "Numeric (Other)"

    # 2. Calculate Skewness
    sk = round(skew(current_series.dropna()), 2) if len(current_series.dropna()) > 2 else 0

    # 3. Calculate Outliers (IQR Method)
    outlier_count = 0
    if len(current_series.dropna()) > 1:
        q1 = current_series.quantile(0.25)
        q3 = current_series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = ((current_series < lower_bound) | (current_series > upper_bound)).sum()

    # 4. Consolidate everything into the dictionary
    stats[col] = {
        "Variable Type": var_type,
        "Unique Values": current_series.nunique(),
        "Min": current_series.min(),
        "Max": current_series.max(),
        "Mean": round(current_series.mean(), 2),
        "Median": round(current_series.median(), 2) if not current_series.empty else 'N/A',
        #"Mode": current_series.mode()[0] if not current_series.mode().empty else 'N/A',
        "Std Dev": round(current_series.std(), 2) if len(current_series.dropna()) > 1 else 'N/A',
        "Skew": sk,
        "Outlier Count": outlier_count
    }

# 5. Create DataFrame and Transpose so columns are rows for better readability
df_stats_report = pd.DataFrame(stats).T

display(HTML(f"<h3>{title}</h3>"))
display(df_stats_report)
print('\n\n')


title = "Time Variable Analysis"

time_stats = {}


for col in df_time.columns:
    current_series = df_time[col]
    time_stats[col] = {
        "Earliest Date": current_series.min(),
        "Latest Date": current_series.max(),
        "Time Range": (current_series.max() - current_series.min()).days}

df_time_stats_report = pd.DataFrame(time_stats).T

display(HTML(f"<h3>{title}</h3>"))
display(df_time_stats_report)
print('\n\n')


title = "Data Validation: Non-Zero/Positive Check"

# 1. Identify invalid rows
invalid_qty = df[df['Quantity'] <= 0]
invalid_price = df[df['UnitPrice'] <= 0]

# 2. Create the Validation Report
validation_data = {
    "Column": ["Quantity", "UnitPrice"],
    "Min Value Found": [df['Quantity'].min(), df['UnitPrice'].min()],
    "Invalid Records (<= 0)": [len(invalid_qty), len(invalid_price)],
    "Status": ["Pass" if len(invalid_qty) == 0 else "Fail",
               "Pass" if len(invalid_price) == 0 else "Fail"]
}

validation_report = pd.DataFrame(validation_data)

# 3. Display with Title
display(HTML(f"<h3>{title}</h3>"))
display(validation_report)
print('\n\n')


title = "Feature Engineering"

# 1. Store state before change
original_shape = df.shape
original_cols = set(df.columns)

# 2. Perform Feature Engineering
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# 3. Identify what changed
new_cols = list(set(df.columns) - original_cols)
col_names = ", ".join(new_cols) if new_cols else "None"

# 4. Define the report data
fe_report = pd.DataFrame({
    "Step": ["Feature Engineering"],
    "New Column(s) Created": [col_names],
    "Original Dimension": [f"{original_shape[0]} rows x {original_shape[1]} columns"],
    "New Dimension": [f"{df.shape[0]} rows x {df.shape[1]} columns"],
    "Net Column Change": [df.shape[1] - original_shape[1]]
})

# 5. Display with a Title
display(HTML("<h3>Feature Engineering: Final Shape Report</h3>"))
display(fe_report)
print('\n\n')

df.head()

**Average order Spend**

In [ ]:
average_order_spend = df.groupby('InvoiceNo')['Revenue'].sum().mean()
print(f"Average Order Spend: ${average_order_spend:.2f}")

**Customer Distribution By Country**

In [ ]:
customer_distribution = df.groupby('Country')['CustomerID'].nunique().sort_values(ascending=False).reset_index().rename(columns={'CustomerID': 'Customer Count Per Country'})
print(f"Customer distribution by country:\n\n{customer_distribution}")
print('\n\n Customer distribution by country:\n')
customer_distribution

**Distilling Datetime into Month**

In [ ]:
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

df.head()

monthly_revenue = df.groupby('InvoiceMonth')['Revenue'].sum().reset_index().rename(columns={'Revenue': 'Monthly Sales'})
monthly_revenue.head()

**Monthly Revenue Visualization**

In [ ]:
monthly_revenue['InvoiceMonth'] = monthly_revenue['InvoiceMonth'].astype(str)

plt.figure(figsize=(12, 6))
sns.lineplot(x='InvoiceMonth', y='Monthly Sales', data=monthly_revenue, marker='o', color='green')
plt.title('Monthly Revenue Trend', fontsize=16)
plt.grid(True)
plt.xticks(rotation=45)
plt.xlabel('Month')
plt.ylabel('Monthly Sales')
plt.tight_layout()
plt.show()

**Cohort Analysis**

**Cohort Analysis: A Deep Dive**

**1. What is Cohort Analysis?**

Cohort Analysis is a powerful analytical technique that groups users based on a shared characteristic (a 'cohort') over a defined period and tracks their behavior over time. Instead of looking at individual customer journeys, it provides a macroscopic view of customer segments, revealing trends and patterns that might otherwise be obscured by traditional aggregated metrics.

In our case, a 'cohort' is defined by the month a customer made their first purchase. This allows us to understand how different groups of customers, acquired at different times, behave throughout their lifecycle with the business.

**2. Why is Cohort Analysis Important?**

  * **Understanding Customer Lifecycle:** It helps visualize how customer engagement, retention, and value evolve over time.

  * **Measuring Impact:** It can highlight the long-term effects of marketing campaigns, product changes, or customer service initiatives launched at specific times.

  * **Identifying Trends: **It helps identify if newer customer cohorts are more or less engaged/retained than older ones.

  * **Informing Strategy:** By pinpointing periods of low retention or engagement, businesses can develop targeted strategies to improve customer loyalty and reduce churn.

**3. How Cohort Analysis was Implemented in this Notebook:**

The Cohort Analysis in this notebook followed these key steps:

  * **a. Defining Cohorts (Cohort Month):** * For each unique customer (CustomerID), we identified the month of their very first transaction using df.groupby(['CustomerID'])['InvoiceMonth'].min(). This 'first purchase month' became their Cohort Month.

  * **b. Calculating Cohort Index:** * To track how many months had passed since a customer's Cohort Month for any given transaction, we calculated the Cohort Index. * This was done by comparing the InvoiceDate (month and year) of each transaction with the customer's Cohort Month. * df['Cohort Index'] = years_diff * 12 + months_diff + 1 where years_diff and months_diff represent the difference between the transaction date and the Cohort Month. * An index of 1 means the transaction occurred in the same month as their first purchase.

  * **c. Counting Unique Customers per Cohort and Period:** * We then grouped the data by Cohort Month and Cohort Index and counted the number of unique customers (CustomerID.nunique()) within each group (cohort_counts). This tells us how many customers from a specific acquisition cohort were still active in subsequent months.

  * **d. Pivoting for the Retention Matrix:** * The cohort_counts data was transformed into a pivot table (cohort_pivot), with Cohort Month as rows and Cohort Index as columns. This structure is ideal for visualizing retention.

  * **e. Calculating Retention Rate:** * The retention rate for each period was calculated by dividing the number of active customers in each Cohort Index by the initial size of that Cohort Month (the number of customers in Cohort Index = 1). * retention_matrix = cohort_pivot.divide(cohort_size, axis=0)

  * **f. Visualizing the Retention Heatmap:** * A heatmap was generated (sns.heatmap(retention_matrix, annot=True, cmap='YlGnBu', fmt='.0%')) to visually represent the retention rates. The fmt='.0%' ensured that the percentages were displayed clearly.

**4. Key Insights from the Retention Heatmap**

Looking at the generated heatmap, we can observe:

  * **Initial Drop-off:** There's a significant drop-off in customer retention from the first month (Cohort Index 1) to the second month (Cohort Index 2) for almost all cohorts. This is common in many businesses and highlights the importance of early engagement strategies.

  * **Stabilization:** After the initial drop, retention rates tend to stabilize or decline more gradually over subsequent months (Cohort Index 3 onwards).

  * **Cohort Performance Variation:** While there's a general trend, some cohorts might show slightly better or worse retention than others. For example, the 2023-12 cohort shows a retention of 40% at Cohort Index 12, which is quite strong compared to other cohorts at similar stages, suggesting a potentially successful acquisition strategy or customer type for that month.

  * **Long-Term Engagement:** We can see that a portion of customers continue to make purchases even 12-13 months after their initial acquisition, indicating a base of loyal customers.

**5. Actionable Recommendations from Cohort Analysis**

  * **Improve Onboarding:** Focus on the first 1-2 months after acquisition. Implement strong onboarding programs, welcome offers, and personalized communication to mitigate the initial churn.

  * **Identify Best-Performing Cohorts:** Analyze what made the most retained cohorts (e.g., 2023-12) successful. Were there specific marketing campaigns, product launches, or external factors that contributed to their higher retention? Replicate those strategies.

  * **Re-engagement Campaigns:** For cohorts showing declining retention, targeted re-engagement campaigns (e.g., special discounts, personalized recommendations) can be developed to bring back inactive customers.

  * **Monitor Trends:** Continuously monitor the retention heatmap to quickly identify any new cohorts that are performing poorly and address the underlying issues in a timely manner

In [ ]:
cohort_data = df.groupby(['CustomerID'])['InvoiceMonth'].min().rename('Cohort Month').reset_index()
cohort_data.name = 'Cohort Month'
cohort_data

**Merging Cohort data with the Original Dataframe**

In [ ]:
cohort_data['CustomerID'] = cohort_data['CustomerID'].astype(str)
df = df.merge(cohort_data, left_on='CustomerID', right_on='CustomerID')
df.head()

Invoice_year, Invoice_month = df['InvoiceDate'].dt.year, df['InvoiceDate'].dt.month
cohort_year, cohort_month = df['Cohort Month'].dt.year, df['Cohort Month'].dt.month

years_diff = Invoice_year - cohort_year
months_diff = Invoice_month - cohort_month

df['Cohort Index'] = years_diff * 12 + months_diff + 1
df.head()

In [ ]:
cohort_counts = df.groupby(['Cohort Month', 'Cohort Index'])['CustomerID'].nunique().reset_index()

cohort_pivot = cohort_counts.pivot_table(index='Cohort Month', columns='Cohort Index', values='CustomerID')
#cohort_pivot = cohort_counts.pivot(index='Cohort Month', columns='Cohort Index', values='CustomerID')
cohort_pivot

**Retention Matrix**

In [ ]:
cohort_size = cohort_pivot.iloc[:,0]
retention_matrix = round(cohort_pivot.divide(cohort_size, axis=0),2)
retention_matrix

**Retention Heat-Map**

In [ ]:
plt.figure(figsize=(10, 8))
#sns.heatmap(retention_matrix, annot=True, cmap='YlGnBu', )
sns.heatmap(retention_matrix, annot=True, cmap='YlGnBu', fmt='.0%', annot_kws={'size': 9})
plt.title('Customer Retention Heatmap', fontsize=16)
plt.xlabel('Months Since the First Purchase ')
plt.ylabel('Cohort Month')
plt.show()

**RFM Analysis: A Deep Dive**

**1. What is RFM Analysis?**

RFM (Recency, Frequency, Monetary) analysis is a marketing technique used to quantitatively rank and group customers based on their past buying behavior. It segments customers into homogeneous groups to identify a company's best customers and target specific customer groups with marketing campaigns.

  * **Recency (R):** How recently did the customer make a purchase? (e.g., days since last purchase). More recent customers are generally more responsive.

  * **Frequency (F):** How often do they purchase? (e.g., number of transactions). Frequent buyers are more engaged and loyal.

  * **Monetary (M):** How much money do they spend? (e.g., total spend). High-spending customers are more valuable.

**2. Why is RFM Analysis Important?**

* **RFM analysis is crucial for:**

    * **Identifying High-Value Customers:** Pinpointing your most valuable and loyal customers (e.g., VIPs, Champions).

    * **Targeted Marketing:** Tailoring marketing strategies to specific customer segments. For example, re-engagement campaigns for customers with high recency but low frequency.

    * **Improving Retention:** Understanding customer behavior helps in predicting churn and implementing proactive retention strategies.

    * **Optimizing Marketing Spend:** Focusing resources on segments that are most likely to respond positively, leading to higher ROI.

    * **Personalization:** Delivering more relevant offers and communications based on individual purchasing patterns.

**3. How RFM Analysis was Implemented in the Notebook:**
Our RFM analysis involved several key steps, building upon the cleaned transactional data:

  * **a. Defining the reference_date:** * To calculate Recency, we needed a consistent point in time from which to measure how 'recent' a customer's last purchase was. This was defined as one day after the absolute latest InvoiceDate in the entire dataset (reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)). * Adding a day ensures that even the most recent purchases have a Recency value of at least 1, preventing zero values that can complicate calculations or interpretations.

  * **b. Aggregating RFM Metrics:** * We grouped the DataFrame by CustomerID and applied aggregation functions to calculate R, F, and M: * Recency: Calculated as the difference in days between the reference_date and each customer's InvoiceDate.max() (their last purchase date). * Frequency: Determined by counting the number of unique InvoiceNo for each customer ('InvoiceNo': 'nunique'). This counts distinct transactions rather than individual items. * Monetary: Calculated as the sum() of the Revenue (the engineered column) for each customer. * The result was stored in the rfm DataFrame with columns CustomerID, Recency, Frequency, and Monetary.

  * **c. Exploring RFM Distribution:** * Descriptive statistics (rfm.describe().T) were generated to understand the range and distribution of each RFM metric. This step provided initial insights into customer spending habits and activity levels.

  * **d. Scaling RFM Features:** * For the subsequent clustering analysis (K-Means), it's crucial that all features contribute equally. Since Recency, Frequency, and Monetary values often operate on different scales, they were standardized using StandardScaler from sklearn.preprocessing. This transformed the data to have a mean of 0 and a standard deviation of 1.

**4. Key Insights from RFM Analysis (Pre-Clustering):**

Before even applying clustering, the raw RFM metrics reveal important patterns:

  * **Recency Distribution:** Observing the range of recency_days helps understand how active the customer base is. A high maximum recency indicates a significant portion of inactive customers.

  * **Frequency Variance:** The mean and standard deviation of frequency show how often customers generally purchase. A high standard deviation suggests a mix of very frequent and very infrequent buyers.

  * **Monetary Skewness:** The monetary value is often skewed, with a few high-value customers driving a large portion of the revenue. This highlights the importance of identifying and nurturing these top spenders.

By quantifying these three dimensions of customer behavior, RFM analysis lays the groundwork for powerful customer segmentation, allowing businesses to move beyond a one-size-fits-all approach to customer engagement.

**RFM - Code Breakdown**

The code in the next code cell combines a pandas aggregation with a time-math object:

* **df['InvoiceDate'].max()**

  *  **What it does:** It scans the entire InvoiceDate column and finds the most recent (latest) date and time.

  * **The Logic:** In a static dataset (like a CSV from last year), the "current date" isn't actually today's real-world date; it is the last day the data was recorded.
    Result: A single Timestamp (e.g., 2011-12-09 12:50:00).

* **pd.Timedelta(days=1)**

  * **What it does**: This creates a standard unit of time duration in pandas.

  * **The Logic:** It represents exactly 24 hours.

* **The Addition (+)**

  * **What it does:** It adds that 24-hour window to the latest transaction date.

  * **Why add a day?**

    * **Avoiding Zeroes:** If a customer bought something at the very last second of the dataset, their "Days Since Last Purchase" would be 0.

    * **Math Safety:** Many statistical formulas (like Recency in RFM analysis) work better if the minimum value is at least 1. By adding a day, the most recent customer will show as "1 day ago" instead of "0 days ago."



In [ ]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Recency Reference Date: {reference_date}")

**Explaining the rfm line of code**

* **rfm = df.groupby('CustomerID').agg({'InvoiceDate': lambda x: (reference_date - x.max()).days, 'InvoiceNo': 'nunique', 'Revenue': 'sum'}).reset_index()**

  * In simple terms, this code creates an RFM (Recency, Frequency, Monetary) table. It looks at every customer's history and calculates three specific numbers: how recently they shopped, how often they shop, and how much money they spent in total.

* **Detailed Breakdown:**
The code uses the .agg() (aggregation) function to perform three different math operations on three different columns simultaneously, all grouped by the CustomerID.

  * **1. Recency: lambda x: (reference_date - x.max()).days**

    * **The Column: InvoiceDate**    
    * **x.max():** Finds the date of that specific customer's most recent purchase.

    * **reference_date** - ...: Subtracts that latest purchase date from your "Snapshot Date."

    * **.days:** Converts the result into a simple whole number (e.g., "42 days ago").

    * **The Goal:** Lower numbers are better; it means the customer shopped very recently.

  * **2. Frequency: 'InvoiceNo': 'nunique'**

    * **The Column: InvoiceNo**

    *  **nunique:** Counts the number of unique invoice numbers for that customer.

    * **The Goal:** This ignores how many individual items they bought and focuses on how many separate visits or orders they made.

  * **3. Monetary: 'Revenue': 'sum'**

    * **The Column: Revenue**

    * **sum:** Adds up every dollar spent across all their transactions.
    
    * **The Goal:** This identifies your "Big Spenders."

  * **4. The Final Touch: .reset_index()**

    * **The Action:** By default, groupby turns the CustomerID into the "Index" (the bold labels on the left).

    * **The Result:** This moves CustomerID back into its own regular column, making the table look like a standard spreadsheet that is ready for further analysis or plotting.




In [ ]:
rfm = df.groupby('CustomerID').agg({'InvoiceDate': lambda x: (reference_date - x.max()).days, 'InvoiceNo': 'nunique', 'Revenue': 'sum'}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm

In [ ]:
rfm.columns = ['CustomerID', 'recency_days', 'frequency','monetary']


display(rfm.describe().T)

print('\n\n')
rfm.head()

**Data Preprocessing**

**Scaling and Normalization**

In [ ]:
from sklearn.preprocessing import StandardScaler


X = rfm[['recency_days', 'frequency','monetary']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

**Modelling - Classification/Clustering**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertia_values = []
silhouette_scores = []

K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertia_values.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))


**Elbow Method Plot**

This plot helps identify the optimal number of clusters for K-Means. It looks for the 'elbow point' where the rate of decrease in inertia (the sum of squared distances of samples to their closest cluster center) significantly slows down. While the plot shows inertia decreasing as K increases, there isn't a very clear, sharp elbow in this particular visualization. However, we can observe a more gradual change after K=3 or K=4, suggesting that 3 or 4 might be a reasonable number of clusters to consider.

In [ ]:
plt.figure(figsize=(12, 8))
plt.plot(K_range, inertia_values, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

**Silhouette Score Plot**

The Silhouette Score plot (as shown in cell the code cell below) is a critical for evaluating the quality of clusters formed by algorithms like K-Means and for helping to determine the optimal number of clusters (K).

* **What it Measures:** The Silhouette Score quantifies how similar an object is to its own cluster compared to other clusters. Scores range from -1 to 1:
        
  * **Close to 1:** Indicates that the object is well matched to its own cluster and poorly matched to neighboring clusters.

  * **Close to 0:** Indicates that the object is on or very close to the decision boundary between two neighboring clusters.

  * **Close to -1:** Indicates that the object might have been assigned to the wrong cluster.

* **Interpretation:** When analyzing the plot, we look for the value of K that yields the highest average Silhouette Score. A higher score generally implies better-defined, more distinct, and more separated clusters. In your plot, you would observe the average silhouette score for each K value from 2 to 10, and the K corresponding to the peak score would be considered the most appropriate choice.


In [ ]:
plt.figure(figsize=(12,8))
plt.plot(K_range, silhouette_scores, marker='o', linestyle='--')
plt.title('Silhouette Score for Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

**KElbowVisualizer**

This is a convenient tool that automates the Elbow Method to help determine the optimal number of clusters for K-Means. It plots the 'distortion score' (which is essentially the inertia, or sum of squared distances of samples to their closest cluster center) for a range of k values.

To interpret this plot, one would look for a point where the rate of decrease in the distortion score significantly slows down, forming an 'elbow' shape. This 'elbow' point suggests a good trade-off between minimizing distortion and keeping the number of clusters manageable.

In [ ]:
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

model = KMeans(random_state=42, n_init=10)
visualizer = KElbowVisualizer(model, k=(2,10), force_model=True)
visualizer.fit(X_scaled)
_ = visualizer.show() # Assign the return value to a dummy variable to suppress output

In [ ]:
final_k = 4
kmeans = KMeans(n_clusters=final_k, random_state=42)
kmeans.fit(X_scaled)
rfm['cluster'] = kmeans.labels_
rfm.head()

In [ ]:
cluster_profiles = rfm.groupby('cluster').agg({'CustomerID': 'count', 'recency_days': 'mean', 'frequency': 'mean', 'monetary': 'mean'}).rename(columns={'CustomerID' : 'Number of Customers', 'frequency': 'Unique Transactions'}).round(1)

cluster_profiles

**Cluster Profiles**

 Here are some meaningful profiles for each cluster:

 *  **Cluster 0: Loyal Customers**

    * **Description:** This cluster, comprising 1367 customers, shows moderate recency (last purchase 47.1 days ago), high frequency with an average of 6.3 unique transactions, and a substantial monetary value of $269,730.0. These customers are consistent purchasers and contribute significantly to revenue.

    * **Actionable Insight:** Reward their loyalty, encourage repeat purchases with personalized offers, and consider upsell/cross-sell opportunities.

  * **Cluster 1: New/Occasional Customers**

    * **Description:** This is the largest cluster with 1733 customers. They have moderate recency (last purchase 45.0 days ago) but lower frequency (2.4 unique transactions) and lower monetary value ($99,169.9). They might be newer customers or those who purchase infrequently.

    * **Actionable Insight:** Focus on engagement strategies to increase their purchase frequency and average order value. Provide onboarding experiences for new customers or reminders for occasional buyers.

  * **Cluster 2: At-Risk/Churned Customers**

    * **Description:** This cluster, with 719 customers, is characterized by high recency (last purchase 186.8 days ago), indicating a long time since their last interaction. Their frequency (2.9 unique transactions) and monetary value ($125,700.8) are moderate, but the key indicator is their high recency.

    * **Actionable Insight:** Implement win-back campaigns with special discounts or personalized recommendations to re-engage them before they are fully churned.

  * **Cluster 3: Champions/VIPs**

    * **Description:** This cluster of 553 customers represents your most valuable segment. They have excellent recency (last purchase 41.5 days ago), very high frequency (11.2 unique transactions), and the highest monetary value ($490,713.6). These customers are your best, most frequent, and highest-spending clients.

    * **Actionable Insight:** Provide exclusive offers, VIP treatment, early access to new products, and seek their feedback to maintain satisfaction and advocacy. They are likely to be brand evangelists.

In [ ]:
cluster_names = {0:'Loyal Customers', 1: 'New/Occasional Customers', 2: 'At-Risk/Churned Customers', 3: 'Champions/VIP'}

cluster_profiles['Cluster Name'] = cluster_profiles.index.map(cluster_names)

cluster_profiles

In [ ]:
cluster_profiles['Segment'] = cluster_profiles.index.map(cluster_names)


In [ ]:
df_plot = cluster_profiles.copy()

df_plot[['recency_days', 'Unique Transactions', 'monetary']] = df_plot[['recency_days', 'Unique Transactions', 'monetary']].apply(lambda x: x / x.max())

df_plot.set_index('Segment')[['recency_days', 'Unique Transactions', 'monetary']].plot(kind='bar', figsize=(12, 8))

plt.title('Normalized Cluster Profiles/Customer Segments')
plt.ylabel('Normalized Value')
plt.xticks(rotation=0)
plt.legend(['Recency', 'Unique Transactions', 'Monetary'], bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

In [ ]:
rfm[['CustomerID', 'cluster', 'recency_days', 'frequency', 'monetary']].to_csv('rfm_clusters.csv', index=False)
print('rfm_cluster File Saved')